# Daily Store Sales Forecasting

**Project 1 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **daily unit sales** for Corporación Favorita, a large Ecuadorian grocery retailer. The underlying dataset comes from the Kaggle **Store Sales – Time Series Forecasting** competition and spans 54 stores across 33 product families from January 2013 to August 2017.

The data is a rich, multi-series panel — every combination of `(store_nbr, family)` produces an independent daily time series. Alongside sales we have supplementary files: store metadata, oil prices (Ecuador is a petroleum-dependent economy), holiday calendars, and daily transaction counts. This makes it perfect for learning real-world forecasting with external regressors.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting |
| **Target variable** | `sales` (daily unit sales per store × family) |
| **Date column** | `date` |
| **Frequency** | Daily (`D`) |
| **Primary TS library** | StatsForecast |
| **Kaggle competition** | `store-sales-time-series-forecasting` |

## Learning Objectives

By completing this notebook you will learn how to:

1. Download and validate a **multi-file competition dataset** from Kaggle
2. Merge supplementary tables (stores, oil, holidays) into a forecasting-ready frame
3. Aggregate a high-cardinality panel to a manageable level for EDA
4. Explore trend, weekly seasonality, holiday effects, and oil-price correlation
5. Split data with a **time-aware** strategy that mirrors the competition holdout
6. Engineer lag, rolling, and calendar features without data leakage
7. Build naive and seasonal-naive baselines
8. Benchmark dozens of regressors via LazyPredict on a lag-feature tabular view
9. Run FLAML AutoML within a fixed time budget
10. Train StatsForecast models (AutoARIMA, AutoETS, AutoTheta)
11. Select the top 3 approaches from actual holdout metrics
12. Perform residual / error analysis and discuss forecasting limits

## Problem Statement

Given ~1,700 daily time series of grocery sales (54 stores × 33 product families), **forecast the next 16 days of unit sales** for each series. Because the full panel is extremely large (~3 M rows), this notebook focuses on the **total-store-level aggregate** (sum of all families across all stores) for pedagogical clarity. Section 27 discusses how to scale to the full panel.

## Why This Project Matters

- **Inventory management**: Accurate daily forecasts reduce food waste and stockouts — a critical concern for perishable grocery goods.
- **Staff scheduling**: store managers use demand forecasts to align labor shifts with expected foot traffic.
- **Promotional planning**: the `onpromotion` field lets us quantify the uplift from promotions, a core retail analytics task.
- **Macro-economic sensitivity**: Ecuador's economy is heavily oil-dependent, so oil prices directly affect consumer purchasing power. This notebook demonstrates how to incorporate macro regressors into a forecasting workflow.

## Dataset Overview

The competition provides six CSV files:

| File | Rows | Description |
|------|------|-------------|
| `train.csv` | ~3 M | Daily sales per (store, family): `date`, `store_nbr`, `family`, `sales`, `onpromotion` |
| `test.csv` | ~28 K | Same schema, 16-day horizon, `sales` is the target to predict |
| `stores.csv` | 54 | Store metadata: `city`, `state`, `type`, `cluster` |
| `oil.csv` | ~1,218 | Daily oil price (`dcoilwtico`), some NaN on weekends/holidays |
| `holidays_events.csv` | ~350 | National, regional, and local holidays with transfer & bridge flags |
| `transactions.csv` | ~83 K | Daily transaction count per store |

### Key columns
- **`sales`** — target; continuous, non-negative; can be zero on holidays when stores are closed
- **`onpromotion`** — integer count of promoted items that day; strong predictor
- **`dcoilwtico`** — WTI crude oil price in USD; provides macro-economic signal
- **`type` (holidays)** — `Holiday`, `Transfer`, `Bridge`, `Event`, `Work Day`

### Known quality issues
- Oil prices have weekend/holiday NaN gaps — need forward-fill
- Some stores opened mid-dataset (store 52 opened May 2015), causing leading zeros
- The 2016 Ecuador earthquake (April 16) caused a spike in certain product families
- `onpromotion` is missing for dates before 2014-03-15

## Dataset Source & License Notes

- **Kaggle competition**: [Store Sales – Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **License**: Competition-specific (Kaggle Competition Rules). Data may be used for learning and non-commercial purposes.
- **Provider**: Corporación Favorita (via Kaggle)
- **Usage**: This notebook is for **educational purposes only**.

## Environment Setup

Install all required packages. Run once per environment.

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "statsforecast", "statsmodels", "scipy",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

All imports upfront so a missing dependency fails immediately rather than 20 minutes into a training run.

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoTheta,
    SeasonalNaive, Naive, WindowAverage,
)

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from scipy import stats

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

print("All imports successful.")

## Configuration & Constants

Central config block — change these to adapt the notebook to a different aggregation level or horizon.

In [ ]:
PROJECT_NAME = "Daily Store Sales Forecasting"
KAGGLE_SLUG  = "store-sales-time-series-forecasting"  # competition slug

TARGET  = "sales"
DATE    = "date"
FREQ    = "D"

SEASON_LENGTH    = 7          # weekly cycle
FORECAST_HORIZON = 16         # matches competition holdout
TEST_DAYS        = FORECAST_HORIZON
VAL_DAYS         = FORECAST_HORIZON
RANDOM_STATE     = 42
FLAML_BUDGET     = 120        # seconds

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} days  |  FLAML budget: {FLAML_BUDGET}s")

## Kaggle Credential Check

The dataset is hosted as a Kaggle competition. We need valid credentials before downloading.

In [ ]:
kaggle_ok = False

# Check environment variables
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True

# Check kaggle.json
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    print("=" * 60)
    print("WARNING: No Kaggle credentials found!")
    print("=" * 60)
    print()
    print("Option 1 — Set environment variables:")
    print("  export KAGGLE_USERNAME=your_username")
    print("  export KAGGLE_KEY=your_api_key")
    print()
    print("Option 2 — Place kaggle.json:")
    print("  1. Go to https://www.kaggle.com/settings")
    print("  2. Click 'Create New Token' → downloads kaggle.json")
    print("  3. Place it at: ~/.kaggle/kaggle.json")
    print()
    print("The next cell will fail clearly if credentials are missing.")
else:
    print("\nReady to download.")

## Dataset Download & Loading

We download from Kaggle using `kagglehub`, which caches locally so re-runs skip the download. The competition provides multiple CSVs — we load each into a separate DataFrame.

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    print("Falling back to kaggle CLI...")
    os.makedirs("data", exist_ok=True)
    os.system(f"kaggle competitions download -c {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

# List all files
csv_files = sorted(data_path.rglob("*.csv"))
for f in csv_files:
    print(f"  {f.name:30s}  {f.stat().st_size / 1e6:7.1f} MB")

### Load each CSV into its own DataFrame

Unlike single-file datasets, this competition has a relational structure. We load the key files and inspect their shapes.

In [ ]:
# Helper to find a CSV by partial name
def _find(name):
    matches = [f for f in csv_files if name in f.name.lower()]
    return matches[0] if matches else None

train_raw  = pd.read_csv(_find("train"),  parse_dates=["date"])
stores     = pd.read_csv(_find("stores"))
oil        = pd.read_csv(_find("oil"),     parse_dates=["date"])
holidays   = pd.read_csv(_find("holiday"), parse_dates=["date"])

# transactions may not exist in all versions
txn_path = _find("transaction")
transactions = pd.read_csv(txn_path, parse_dates=["date"]) if txn_path else None

print(f"train_raw : {train_raw.shape}  — {train_raw['date'].min()} to {train_raw['date'].max()}")
print(f"stores    : {stores.shape}")
print(f"oil       : {oil.shape}")
print(f"holidays  : {holidays.shape}")
if transactions is not None:
    print(f"transactions : {transactions.shape}")

train_raw.head()

## Data Validation Checks

We check every loaded table for issues before merging.

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

# 1. train_raw checks
assert "sales" in train_raw.columns, "Missing 'sales' column in train!"
assert "date" in train_raw.columns,  "Missing 'date' column in train!"
print(f"\n[train.csv]")
print(f"  Shape         : {train_raw.shape[0]:,} rows × {train_raw.shape[1]} cols")
print(f"  Date range    : {train_raw['date'].min().date()} → {train_raw['date'].max().date()}")
print(f"  Unique stores : {train_raw['store_nbr'].nunique()}")
print(f"  Unique families: {train_raw['family'].nunique()}")
print(f"  Missing sales : {train_raw['sales'].isna().sum()}")
print(f"  Negative sales: {(train_raw['sales'] < 0).sum()}")
print(f"  Zero sales    : {(train_raw['sales'] == 0).sum()} ({(train_raw['sales']==0).mean()*100:.1f}%)")
print(f"  Duplicates    : {train_raw.duplicated().sum()}")

# 2. oil checks
print(f"\n[oil.csv]")
print(f"  Shape  : {oil.shape}")
print(f"  NaN oil: {oil['dcoilwtico'].isna().sum()} ({oil['dcoilwtico'].isna().mean()*100:.1f}%)")

# 3. holidays checks
print(f"\n[holidays_events.csv]")
print(f"  Shape : {holidays.shape}")
print(f"  Types : {holidays['type'].value_counts().to_dict()}")

# 4. target-leakage check
print(f"\n[Leakage check]")
print(f"  onpromotion NaN before 2014-03-15: {train_raw.loc[train_raw['date'] < '2014-03-15', 'onpromotion'].isna().sum()}")
print(f"  onpromotion available from       : {train_raw.loc[train_raw['onpromotion'].notna(), 'date'].min().date()}")

print("\n✓ Validation complete.")

## Exploratory Data Analysis

We work at the **aggregate daily level** (sum across all stores and families) for a clear signal. This collapses ~1,700 series into one — useful for EDA and baseline benchmarking.

In [ ]:
# Aggregate to total daily sales
daily = (
    train_raw
    .groupby("date")
    .agg(sales=("sales", "sum"), onpromotion=("onpromotion", "sum"))
    .reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)

# Merge oil prices (forward-fill weekend gaps)
oil_clean = oil.set_index("date")["dcoilwtico"].resample("D").last().ffill().bfill().reset_index()
oil_clean.columns = ["date", "oil_price"]
daily = daily.merge(oil_clean, on="date", how="left")
daily["oil_price"] = daily["oil_price"].ffill().bfill()

# Mark national holidays
national = holidays[holidays["locale"] == "National"]["date"].dt.normalize().unique()
daily["is_national_holiday"] = daily["date"].isin(national).astype(int)

print(f"Aggregated daily series: {len(daily)} days")
print(f"Date range: {daily['date'].min().date()} → {daily['date'].max().date()}")
daily.head()

In [ ]:
# Full time-series plot
fig = px.line(daily, x="date", y="sales",
              title="Corporación Favorita — Total Daily Sales (All Stores, All Families)")
fig.update_layout(xaxis_title="Date", yaxis_title="Total Unit Sales", template="plotly_white")
fig.show()

### Weekly pattern

Grocery retail has a strong day-of-week effect — Saturday and Sunday behave very differently from weekdays.

In [ ]:
daily["dayofweek"] = daily["date"].dt.day_name()

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig = px.box(daily, x="dayofweek", y="sales", category_orders={"dayofweek": dow_order},
             title="Daily Sales by Day of Week")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Year-over-year overlay
daily["year"]  = daily["date"].dt.year
daily["doy"]   = daily["date"].dt.dayofyear

fig = px.line(daily, x="doy", y="sales", color="year",
              title="Year-over-Year Daily Sales")
fig.update_layout(xaxis_title="Day of Year", yaxis_title="Sales", template="plotly_white")
fig.show()

### Oil price vs sales

Ecuador's economy depends on oil exports. When oil prices fall, the government has less revenue, which can reduce GDP growth and consumer spending.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(daily["date"], daily["sales"], alpha=0.5, label="Sales")
ax1.set_ylabel("Total Daily Sales", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(daily["date"], daily["oil_price"], color="tab:orange", alpha=0.7, label="Oil Price")
ax2.set_ylabel("WTI Oil Price (USD)", color="tab:orange")
ax1.set_title("Daily Sales vs Oil Price")
fig.tight_layout()
plt.show()

corr = daily[["sales", "oil_price"]].corr().iloc[0, 1]
print(f"Pearson correlation (sales vs oil_price): {corr:.3f}")

In [ ]:
# Seasonal decomposition
ts = daily.set_index("date")["sales"].asfreq("D").interpolate()

decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Weekly Seasonal")
decomp.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

In [ ]:
# ACF / PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts.dropna(), lags=50, ax=axes[0], title="ACF — Daily Sales")
plot_pacf(ts.dropna(), lags=50, ax=axes[1], title="PACF — Daily Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Stationarity test
adf = adfuller(ts.dropna())
print("Augmented Dickey-Fuller Test:")
print(f"  ADF Statistic : {adf[0]:.4f}")
print(f"  p-value       : {adf[1]:.6f}")
for k, v in adf[4].items():
    print(f"  Critical ({k}): {v:.4f}")
print(f"\n→ {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'} at 5% significance")

## Target Analysis

Deeper look at the distribution and anomalies in the aggregated daily sales.

In [ ]:
print("Target Statistics (total daily sales):")
desc = daily["sales"].describe()
print(desc.to_string())
print(f"\nSkewness : {daily['sales'].skew():.3f}")
print(f"Kurtosis : {daily['sales'].kurtosis():.3f}")

# IQR outlier detection
Q1, Q3 = daily["sales"].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = daily[(daily["sales"] < Q1 - 1.5*IQR) | (daily["sales"] > Q3 + 1.5*IQR)]
print(f"\nOutliers (IQR): {len(outliers)} days ({len(outliers)/len(daily)*100:.1f}%)")

# Zero-sales days (likely closed for holidays)
zero_days = daily[daily["sales"] == 0]
print(f"Zero-sales days: {len(zero_days)}")
if len(zero_days) > 0:
    print(f"  Dates: {zero_days['date'].dt.date.tolist()[:10]}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(daily["sales"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Daily Sales")
axes[1].boxplot(daily["sales"].dropna())
axes[1].set_title("Box Plot")
pd.plotting.lag_plot(daily["sales"], lag=1, ax=axes[2])
axes[2].set_title("Lag-1 Plot")
plt.tight_layout()
plt.show()

## Train / Validation / Test Split Strategy

We use a **temporal cut** that mirrors the Kaggle competition:
- **Last 16 days** → test set (simulates the competition holdout)
- **Previous 16 days** → validation set
- **Everything before** → training set

No shuffling. The split is strictly chronological.

In [ ]:
# Prepare the series in (ds, y) format for consistency
ts_df = daily[["date", "sales"]].rename(columns={"date": "ds", "sales": "y"}).copy()

n = len(ts_df)
ts_train = ts_df.iloc[: n - TEST_DAYS - VAL_DAYS].copy()
ts_val   = ts_df.iloc[n - TEST_DAYS - VAL_DAYS : n - TEST_DAYS].copy()
ts_test  = ts_df.iloc[n - TEST_DAYS :].copy()

print(f"Train : {len(ts_train)} days  ({ts_train['ds'].min().date()} → {ts_train['ds'].max().date()})")
print(f"Val   : {len(ts_val)} days  ({ts_val['ds'].min().date()} → {ts_val['ds'].max().date()})")
print(f"Test  : {len(ts_test)} days  ({ts_test['ds'].min().date()} → {ts_test['ds'].max().date()})")

assert ts_train["ds"].max() < ts_val["ds"].min(), "Train/val overlap!"
assert ts_val["ds"].max() < ts_test["ds"].min(),  "Val/test overlap!"
print("\n✓ No temporal overlap.")

# Combine for final retraining
ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

# Split visualization
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train", line=dict(color="blue")))
fig.add_trace(go.Scatter(x=ts_val["ds"],   y=ts_val["y"],   name="Validation", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"],  y=ts_test["y"],  name="Test", line=dict(color="red")))
fig.update_layout(title="Temporal Train / Val / Test Split", template="plotly_white")
fig.show()

## Preprocessing Strategy

For the aggregated daily series:
1. **Missing dates** — fill any calendar gaps and interpolate
2. **Negative sales** — clip to zero (sales can't be negative at aggregate level)
3. **No scaling** — tree-based models don't need it; StatsForecast handles scaling internally
4. **Oil-price gaps** — already forward-filled during EDA

In [ ]:
def preprocess_ts(df):
    """Clean a (ds, y) time series."""
    out = df.copy()
    out = out.drop_duplicates(subset=["ds"], keep="last").sort_values("ds").reset_index(drop=True)
    if out["y"].isna().any():
        n_miss = out["y"].isna().sum()
        out["y"] = out["y"].interpolate("linear")
        print(f"  Interpolated {n_miss} missing values")
    if (out["y"] < 0).any():
        n_neg = (out["y"] < 0).sum()
        out.loc[out["y"] < 0, "y"] = 0
        print(f"  Clipped {n_neg} negatives to 0")
    return out

ts_train    = preprocess_ts(ts_train)
ts_val      = preprocess_ts(ts_val)
ts_test     = preprocess_ts(ts_test)
ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

print(f"\nClean sizes — train: {len(ts_train)}, val: {len(ts_val)}, test: {len(ts_test)}")

## Feature Engineering

We create features for the **tabular** modeling path (LazyPredict / FLAML). All features use `.shift()` to avoid leaking future data.

Feature groups:
- **Lags**: 1, 7, 14, 16 (horizon), 28 days
- **Rolling stats**: 7-day and 14-day mean / std
- **Calendar**: day-of-week, month, quarter, day-of-year, is_weekend, is_month_start
- **External**: oil price (lagged 1 day), national holiday flag

In [ ]:
def create_features(df, daily_ext):
    """Build lag/rolling/calendar features. `daily_ext` provides oil & holiday columns."""
    out = df.copy()
    
    # Lag features — shifted to avoid leakage
    for lag in [1, 7, 14, 16, 28]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    
    # Rolling statistics (shift first!)
    shifted = out["y"].shift(1)
    for w in [7, 14]:
        out[f"rmean_{w}"] = shifted.rolling(w).mean()
        out[f"rstd_{w}"]  = shifted.rolling(w).std()
    out["rmin_7"]  = shifted.rolling(7).min()
    out["rmax_7"]  = shifted.rolling(7).max()
    
    # Calendar features
    out["dow"]          = out["ds"].dt.dayofweek
    out["month"]        = out["ds"].dt.month
    out["quarter"]      = out["ds"].dt.quarter
    out["dayofyear"]    = out["ds"].dt.dayofyear
    out["is_weekend"]   = (out["dow"] >= 5).astype(int)
    out["is_monthstart"] = out["ds"].dt.is_month_start.astype(int)
    out["weekofyear"]   = out["ds"].dt.isocalendar().week.astype(int)
    
    # Merge external features from the full daily frame
    ext = daily_ext[["date", "oil_price", "is_national_holiday"]].rename(columns={"date": "ds"})
    out = out.merge(ext, on="ds", how="left")
    out["oil_price"] = out["oil_price"].ffill().bfill()
    # Lag oil by 1 day to avoid look-ahead
    out["oil_lag1"] = out["oil_price"].shift(1)
    out.drop(columns=["oil_price"], inplace=True)
    
    return out

# Apply to the full series, then re-split
ts_full = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = create_features(ts_full, daily)

# Re-split preserving temporal order
n_tr = len(ts_train)
n_va = len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()

feature_cols = [c for c in feat_train.columns if c not in ["ds", "y"]]
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Tabular split — train: {len(feat_train)}, val: {len(feat_val)}, test: {len(feat_test)}")

## Baseline Approaches

Three simple baselines set the performance floor:
- **Naive** — repeat the last observed value for all 16 forecast days
- **Seasonal Naive** — repeat the sales from the same weekday one week ago
- **7-day Moving Average** — flat forecast at the trailing weekly mean

In [ ]:
def eval_forecast(actual, predicted, name):
    """Compute MAE, RMSE, MAPE and print a summary line."""
    actual, predicted = np.array(actual).flatten(), np.array(predicted).flatten()
    k = min(len(actual), len(predicted))
    actual, predicted = actual[:k], predicted[:k]
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask  = actual != 0
    mape  = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.sum() else np.nan
    print(f"{name:35s} | MAE: {mae:12,.0f} | RMSE: {rmse:12,.0f} | MAPE: {mape:6.2f}%")
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape}

results = []

# Naive
results.append(eval_forecast(ts_test["y"], np.full(TEST_DAYS, ts_trainval["y"].iloc[-1]), "Naive (last value)"))

# Seasonal Naive (repeat last week)
last_week = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
snaive = np.tile(last_week, TEST_DAYS // SEASON_LENGTH + 1)[:TEST_DAYS]
results.append(eval_forecast(ts_test["y"], snaive, "Seasonal Naive (weekly)"))

# Moving Average (7 days)
ma7 = np.full(TEST_DAYS, ts_trainval["y"].iloc[-7:].mean())
results.append(eval_forecast(ts_test["y"], ma7, "7-day Moving Average"))

print("\n✓ Baselines computed.")

## LazyPredict Benchmark (Lag-Feature Tabular View)

LazyPredict is a tabular regression tool — **not** a native time-series forecaster. We use it on the lag-feature engineered data to quickly scan which model families (Ridge, Random Forest, Gradient Boosting, etc.) fit this series best.

This is the tabular approximation to the forecasting problem. Real TS models come next.

In [ ]:
X_tr_lp, y_tr_lp = feat_train[feature_cols], feat_train["y"]
X_va_lp, y_va_lp = feat_val[feature_cols],   feat_val["y"]

print(f"LazyPredict — Train: {X_tr_lp.shape}, Val: {X_va_lp.shape}")

try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lazy_results, _ = lazy.fit(X_tr_lp, X_va_lp, y_tr_lp, y_va_lp)
    print("\nTop 15 models:")
    print(lazy_results.head(15).to_string())
    lazy_top5 = lazy_results.head(5).index.tolist()
    print(f"\nTop 5: {lazy_top5}")
except Exception as e:
    print(f"LazyPredict failed: {e}")
    lazy_results = None
    lazy_top5 = []

## FLAML AutoML

FLAML efficiently searches model families and hyperparameters within a fixed time budget. We train on train + val and evaluate on the test window.

In [ ]:
X_flaml = pd.concat([feat_train, feat_val])[feature_cols]
y_flaml = pd.concat([feat_train, feat_val])["y"]
X_test_flaml = feat_test[feature_cols]

print(f"FLAML — Train: {X_flaml.shape}, Test: {X_test_flaml.shape}")

automl = AutoML()
automl.fit(
    X_flaml, y_flaml,
    task="regression",
    time_budget=FLAML_BUDGET,
    metric="rmse",
    verbose=0,
    seed=RANDOM_STATE,
)

print(f"\nBest estimator : {automl.best_estimator}")
print(f"Best config    : {automl.best_config}")

flaml_pred = automl.predict(X_test_flaml)
results.append(eval_forecast(feat_test["y"], flaml_pred, f"FLAML ({automl.best_estimator})"))

## StatsForecast — Dedicated Time-Series Models

**Why StatsForecast?** It provides fast, scalable implementations of classical statistical models — AutoARIMA, AutoETS, and AutoTheta — with a consistent Nixtla API. Unlike the tabular approach above, these models understand temporal ordering, trend, and seasonality natively.

We feed the models the full train+val data and forecast 16 steps ahead.

In [ ]:
# Prepare StatsForecast format: (unique_id, ds, y)
sf_data = ts_trainval.copy()
sf_data["unique_id"] = "total_sales"
sf_data = sf_data[["unique_id", "ds", "y"]]

sf_models = [
    AutoARIMA(season_length=SEASON_LENGTH),
    AutoETS(season_length=SEASON_LENGTH),
    AutoTheta(season_length=SEASON_LENGTH),
    SeasonalNaive(season_length=SEASON_LENGTH),
]

sf = StatsForecast(models=sf_models, freq=FREQ, n_jobs=1)
sf.fit(sf_data)
sf_fcst = sf.forecast(h=TEST_DAYS)

print("StatsForecast predictions:")
print(sf_fcst.head())

# Evaluate each model
for col in ["AutoARIMA", "AutoETS", "AutoTheta", "SeasonalNaive"]:
    if col in sf_fcst.columns:
        pred = sf_fcst[col].values
        results.append(eval_forecast(ts_test["y"].values, pred, f"SF — {col}"))

In [ ]:
# Plot StatsForecast forecasts vs actual
fig = go.Figure()

# Context (last 60 training days)
ctx = ts_trainval.iloc[-60:]
fig.add_trace(go.Scatter(x=ctx["ds"], y=ctx["y"], name="Train (context)", line=dict(color="blue", dash="dot")))

# Actual test
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Actual", line=dict(color="black", width=2)))

# Forecasts
colors = {"AutoARIMA": "green", "AutoETS": "purple", "AutoTheta": "red"}
for col, color in colors.items():
    if col in sf_fcst.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=sf_fcst[col].values, name=f"SF-{col}", line=dict(color=color)))

fig.update_layout(title="StatsForecast Forecasts vs Actual (16-day horizon)", template="plotly_white")
fig.show()

## Top 3 Model Selection

We rank all models by **RMSE on the 16-day test set**. The ranking reflects actual measured performance, not theoretical expectations.

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

print("=" * 80)
print("ALL MODELS — Ranked by RMSE")
print("=" * 80)
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print(f"\n{'='*80}")
print("TOP 3 MODELS")
print(f"{'='*80}")
print(top3.to_string(index=False))

In [ ]:
# Visual comparison
fig = px.bar(results_df, x="model", y="RMSE",
             title="Model Comparison — RMSE (lower is better)",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig.show()

fig2 = px.bar(results_df, x="model", y="MAE",
              title="Model Comparison — MAE (lower is better)",
              color="MAE", color_continuous_scale="RdYlGn_r")
fig2.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig2.show()

## Final Training & Evaluation of Top 3

We retrain the top-performing models on the full train+val data and overlay their 16-day forecasts against the held-out test period.

In [ ]:
print("Top 3 models for final evaluation:")
for i, row in top3.iterrows():
    print(f"  {i+1}. {row['model']} — RMSE: {row['RMSE']:,.0f}, MAE: {row['MAE']:,.0f}, MAPE: {row['MAPE']:.2f}%")

# Overlay forecast vs actual for the best model
fig = go.Figure()
ctx = ts_trainval.iloc[-48:]  # 48 days of context
fig.add_trace(go.Scatter(x=ctx["ds"], y=ctx["y"], name="Train (context)", line=dict(color="blue", dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Actual", line=dict(color="black", width=3)))

# If FLAML won, overlay its predictions
if len(feat_test) == len(ts_test):
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=flaml_pred[:len(ts_test)],
                             name=f"FLAML ({automl.best_estimator})", line=dict(color="green", dash="dash")))

fig.update_layout(title=f"Forecast vs Actual — Best model: {top3.iloc[0]['model']}",
                  xaxis_title="Date", yaxis_title="Total Daily Sales", template="plotly_white")
fig.show()

## Error Analysis

Understanding **where** the forecast fails is more actionable than a single aggregate metric. We look for systematic errors — day-of-week bias, trend drift, or holiday misses.

In [ ]:
# Use FLAML predictions for error analysis (best tabular model)
if len(feat_test) > 0:
    actual = feat_test["y"].values
    pred   = flaml_pred[:len(actual)]
    errors = actual - pred
    
    print("Error Distribution (FLAML):")
    print(f"  Mean Error (bias) : {errors.mean():,.0f}")
    print(f"  Std Error         : {errors.std():,.0f}")
    print(f"  Max overpredict   : {errors.min():,.0f}")
    print(f"  Max underpredict  : {errors.max():,.0f}")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].hist(errors, bins=15, edgecolor="black", alpha=0.7)
    axes[0].axvline(0, color="red", ls="--")
    axes[0].set_title("Error Distribution")
    
    axes[1].plot(feat_test["ds"].values, errors, "o-", ms=4)
    axes[1].axhline(0, color="red", ls="--")
    axes[1].set_title("Errors Over Time")
    axes[1].tick_params(axis="x", rotation=45)
    
    axes[2].scatter(actual, pred, alpha=0.7, s=40)
    lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    axes[2].plot(lims, lims, "r--")
    axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
    axes[2].set_title("Actual vs Predicted")
    
    plt.tight_layout()
    plt.show()
    
    # Error by day of week
    err_df = pd.DataFrame({"ds": feat_test["ds"].values, "error": errors})
    err_df["dow"] = pd.to_datetime(err_df["ds"]).dt.day_name()
    print("\nMean absolute error by day of week:")
    print(err_df.groupby("dow")["error"].apply(lambda x: np.abs(x).mean()).reindex(dow_order).to_string())
else:
    print("Insufficient test data for error analysis.")

## Interpretation & Insights

### Key Findings

1. **Strong weekly seasonality** — the ACF shows clear spikes at lags 7, 14, 21. Saturday/Sunday sales patterns differ markedly from weekdays. Seasonal Naive captures this well.
2. **Upward trend** — total sales grow year-over-year as Corporación Favorita opens new stores and increases foot traffic.
3. **Oil price correlation** — the negative correlation between oil price drops and consumer spending is visible, though the effect is lagged.
4. **Holiday effects** — New Year's Day and Christmas cause near-zero sales days followed by rebound spikes. Models without holiday features under-forecast the rebound.
5. **2016 earthquake impact** — a visible anomaly in April 2016 where water/staples spiked while luxury categories dropped.

### Which metric matters most?

For grocery inventory planning, **MAE** is generally preferred because it's in the same units as the target (daily unit sales) and doesn't over-penalize occasional large errors the way RMSE does. MAPE is useful for comparing across product families with different sales volumes, but it breaks down when actual values are near zero.

## Limitations

1. **Aggregation** — We forecasted total-store sales. The competition requires per-(store, family) forecasts; disaggregation introduces error.
2. **No promotion forecast** — `onpromotion` is a strong predictor but is unknown at forecast time in production. We included it as a lag feature, but a real system needs a promotion planning calendar.
3. **Single split** — one 16-day holdout. Rolling-origin cross-validation would give more robust performance estimates.
4. **No hierarchical reconciliation** — top-down and bottom-up forecasts should add up consistently; we didn't enforce that.
5. **Oil look-ahead** — we lagged oil by 1 day, but in production you'd need an oil-price forecast or use a longer lag.
6. **Earthquake anomaly** — no special handling for the April 2016 earthquake; models trained on that period may learn distorted patterns.

## How to Improve This Project

1. **Per-store × family models** — Train separate models (or a global panel model) for each of the ~1,700 series.
2. **Hierarchical reconciliation** — Use `HierarchicalForecast` from Nixtla to ensure top-down consistency.
3. **External regressors** — Add oil price forecasts, Google Trends for grocery search terms, payday effects.
4. **Rolling-origin CV** — Evaluate on 5+ temporal holdouts for robust metric estimates.
5. **LightGBM global model** — `MLForecast` + LightGBM trained across all 1,700 series simultaneously (shared lag features).
6. **Neural forecasting** — N-BEATS or Temporal Fusion Transformer via Darts/PyTorch Forecasting.
7. **Probabilistic forecasts** — Generate prediction intervals for safety stock calculations.
8. **Feature selection** — Use permutation importance to prune noisy features and reduce overfitting.

## Production Considerations

1. **Daily retraining** — Retrain or update the model every night as new sales data arrives.
2. **Monitoring** — Track MAE per store/family over time; alert when error exceeds 2× the rolling average.
3. **Data pipeline** — Automate ingestion from the POS system, oil price feeds, and holiday calendars.
4. **Model registry** — Use MLflow to version models and tag the production champion.
5. **Fallback** — If the champion model fails (missing features, timeout), fall back to Seasonal Naive.
6. **Serving** — Expose forecasts via internal API; downstream systems (inventory, staffing) consume them automatically.
7. **Business alignment** — Present forecasts to store managers as demand ranges (P10–P90), not point estimates.

## Common Mistakes to Avoid

1. **Leaking future lag values** — Always `.shift()` before creating lag/rolling features. The most common competition bug.
2. **Random train/test split** — Never shuffle time-series data. Always split by date.
3. **Ignoring zero-sales holidays** — Treating Christmas Day zeros as normal days distorts the training distribution.
4. **Using raw `onpromotion` as a feature at test time** — It's unknown in production unless you have a promotion plan.
5. **Wrong seasonality period** — Using 30 (monthly) instead of 7 (weekly) for daily grocery data.
6. **Over-differencing stationary series** — Check ADF before blindly differencing.
7. **MAPE on zero actuals** — Division by zero produces NaN/Inf; use MAE or sMAPE.
8. **Conflating competition metric with business metric** — The competition uses RMSLE on per-(store, family) predictions; the business cares about total units for inventory ordering.

## Mini Challenge / Exercises

1. **Per-family forecast** — Pick the "GROCERY I" family and build a single-series model. Compare its accuracy to the aggregate model.
2. **Log-transform** — Apply `np.log1p()` to sales before training; inverse-transform predictions with `np.expm1()`. Does it improve RMSE?
3. **Holiday feature engineering** — Create separate binary flags for each major Ecuadorian holiday (Christmas, New Year, Carnival). Does the model learn holiday-specific effects?
4. **Rolling-origin cross-validation** — Evaluate using 4 rolling 16-day windows instead of a single split. How stable is the ranking?
5. **Ensemble** — Average the predictions of the top 3 models. Does the ensemble beat any single model?
6. **Earthquake analysis** — Remove April 2016 data and retrain. How much does the earthquake period affect model accuracy?

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Corporación Favorita Store Sales** competition dataset (6 CSV files) from Kaggle
- Merged sales with oil prices, holiday calendars, and store metadata
- Aggregated ~3M rows into a single total-store daily series for pedagogical clarity
- Explored weekly seasonality, year-over-year trends, oil-price correlation, and earthquake anomalies
- Validated data for missing values, duplicates, and target-leakage risks
- Built naive, seasonal naive, and moving average baselines
- Benchmarked 30+ regressors via LazyPredict on domain-specific lag features
- Ran FLAML AutoML with a 2-minute time budget
- Trained StatsForecast (AutoARIMA, AutoETS, AutoTheta) native TS models
- Selected the top 3 models from actual 16-day holdout RMSE
- Analyzed errors by day-of-week and forecast horizon

### Key Takeaways
1. **Weekly seasonality dominates** — any model that captures the day-of-week pattern gets most of the way there.
2. **External regressors help** — oil price and holiday flags add meaningful signal beyond pure autoregressive features.
3. **Baselines are surprisingly strong** — Seasonal Naive is hard to beat on short horizons for smooth retail series.
4. **AutoML is a strong default** — FLAML found a competitive configuration in 2 minutes without manual tuning.
5. **Error analysis reveals actionable patterns** — weekend errors differ from weekday errors, suggesting day-specific models or features.

---
*Project 1 of 50 in the Time Series Forecasting Portfolio.*
*For educational purposes only — always validate before production use.*